# Efeito Dominó no Mercado de Transferências
### Análise exploratória
**Objetivo:** Explorar os dados de clubes, jogadores e transferências para identificar sinais brutos do *prêmio do vendedor* e orientar a escolha metodológica.

**Membros:**

1.
2. Leticia Ribeiro Miranda - 2021095686
3. 
4. 

**Estrutura do notebook:**
1. Imports
2. Carga e integração dos dados (JSONL → DataFrame unificado)
3. Limpeza e engenharia de variáveis
4. Análise descritiva das transferências
5. Prêmio bruto: fee vs. valor de mercado
6. Análise do "prêmio do vendedor": clubes que venderam e depois compraram
7. Análise de rede: grafo de transferências
8. Perfil dos jogadores transferidos
9. Conclusões e próximos passos (TP2)

## Imports

In [5]:
import json
import re
import warnings
import glob  
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import networkx as nx
from collections import defaultdict

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120


DATA_DIR = '../data'
clubs_files     = glob.glob(os.path.join(DATA_DIR, '*-clubs.jsonl'))
players_files   = glob.glob(os.path.join(DATA_DIR, '*-players.jsonl'))
transfers_files = glob.glob(os.path.join(DATA_DIR, '*-transfers.jsonl'))
competitions_files = glob.glob(os.path.join(DATA_DIR, '*-competitions.jsonl'))

def load_jsonl_to_df(file_list):
    if not file_list:
        print("Aviso: Nenhum arquivo encontrado para o padrão solicitado.")
        return pd.DataFrame()
    
    dfs = []
    for file in file_list:
        with open(file, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]
        
        df_temp = pd.DataFrame(data)
        
        # Extrai de forma limpa apenas o nome do arquivo (ex: '2023-bundesliga')
        base_name = os.path.basename(file)
        liga_name = (base_name.replace('-transfers.jsonl', '')
                              .replace('-clubs.jsonl', '')
                              .replace('-players.jsonl', '')
                              .replace('-competitions.jsonl', ''))
        
        df_temp['source_league'] = liga_name
        dfs.append(df_temp)
        
    return pd.concat(dfs, ignore_index=True)

df_clubs     = load_jsonl_to_df(clubs_files)
df_players   = load_jsonl_to_df(players_files)
df_transfers = load_jsonl_to_df(transfers_files)
df_competitions = load_jsonl_to_df(competitions_files)

print(f"Clubes carregados: {df_clubs.shape[0]} linhas vindas de {len(clubs_files)} arquivos.")
print(f"Jogadores carregados: {df_players.shape[0]} linhas vindas de {len(players_files)} arquivos.")
print(f"Transferências carregadas: {df_transfers.shape[0]} linhas vindas de {len(transfers_files)} arquivos.")
print(f"Competições carregadas: {df_competitions.shape[0]} linhas vindas de {len(competitions_files)} arquivos.")

Clubes carregados: 390 linhas vindas de 21 arquivos.
Jogadores carregados: 11285 linhas vindas de 21 arquivos.
Transferências carregadas: 16215 linhas vindas de 21 arquivos.
Competições carregadas: 1474 linhas vindas de 3 arquivos.
